# UK Housing Affordability Analysis
## Notebook 01: Data Loading & Initial Inspection

**Author:** David  
**Date:** May 2026  
**Stage:** Data Understanding  

### Objective
Load and combine two datasets:
1. UK House Price Index (HM Land Registry) — average prices by region, 1995 to 2025
2. Annual Survey of Hours and Earnings (ONS) — median earnings by region, 2015 to 2025

Combine them to create the foundation for an affordability ratio analysis.

### Data Sources
- UK HPI Average Price: HM Land Registry — Open Government Licence v3.0
- ASHE Table 1: Office for National Statistics — Open Government Licence v3.0

In [1]:
import pandas as pd
import numpy as np
import os
import zipfile
import glob

# ── CHECK ALL FILES ARE ACCESSIBLE ────────────────────────────────────────
raw_path = '../data/raw/'

all_files = os.listdir(raw_path)
print(f"Total files in raw folder: {len(all_files)}")
print("\nFiles found:")
for f in sorted(all_files):
    print(f"  {f}")

Total files in raw folder: 13

Files found:
  .DS_Store
  ashetable12021revised
  ashetable12022revised
  ashetable12023revised
  ashetable12024revised
  ashetable12025provisional
  table12015revised
  table12016revised
  table12017revised
  table12018revised
  table12019revised
  table12020revised
  uk-hpi-average-price-oct2025.csv


In [2]:
# ── LOAD HOUSE PRICE DATA ─────────────────────────────────────────────────

hpi = pd.read_csv('../data/raw/uk-hpi-average-price-oct2025.csv')

print(f"Shape: {hpi.shape}")
print(f"\nColumns: {list(hpi.columns)}")
print(f"\nDate range: {hpi['Date'].min()} to {hpi['Date'].max()}")
print(f"\nUnique regions: {hpi['Region_Name'].nunique()}")
print(f"\nSample of regions:")
print(hpi['Region_Name'].unique()[:20])
print(f"\nFirst 5 rows:")
hpi.head()


Shape: (148275, 7)

Columns: ['Date', 'Region_Name', 'Area_Code', 'Average_Price', 'Monthly_Change', 'Annual_Change', 'Average_Price_SA']

Date range: 1968-04-01 to 2025-10-01

Unique regions: 405

Sample of regions:
['Northern Ireland' 'England' 'Wales' 'Scotland' 'London' 'East Midlands'
 'Yorkshire and The Humber' 'South West' 'West Midlands Region'
 'United Kingdom' 'South East' 'East of England' 'North East'
 'Inner London' 'Outer London' 'North West' 'Greater Manchester'
 'Merseyside' 'Tyne and Wear' 'West Midlands']

First 5 rows:


,Date,Region_Name,Area_Code,Average_Price,Monthly_Change,Annual_Change,Average_Price_SA
0,1968-04-01,Northern Ireland,N92000002,3465,NaN,NaN,NaN
1,1968-04-01,England,E92000001,3218,NaN,NaN,NaN
2,1968-04-01,Wales,W92000004,2732,NaN,NaN,NaN
3,1968-04-01,Scotland,S92000003,2738,NaN,NaN,NaN
4,1968-04-01,London,E12000007,4730,NaN,NaN,NaN


In [3]:
# ── FILTER TO ENGLAND REGIONS 2015-2025 ───────────────────────────────────

# Convert date to datetime
hpi['Date'] = pd.to_datetime(hpi['Date'])

# The 9 official regions of England we want
england_regions = [
    'North East',
    'North West',
    'Yorkshire and The Humber',
    'East Midlands',
    'West Midlands Region',
    'East of England',
    'London',
    'South East',
    'South West'
]

# Filter to England regions and 2015 onwards
hpi_filtered = hpi[
    (hpi['Region_Name'].isin(england_regions)) &
    (hpi['Date'] >= '2015-01-01')
].copy()

# Keep only the columns we need
hpi_filtered = hpi_filtered[['Date', 'Region_Name', 'Average_Price']].copy()

# Extract year and month
hpi_filtered['year'] = hpi_filtered['Date'].dt.year
hpi_filtered['month'] = hpi_filtered['Date'].dt.month

print(f"Rows after filtering: {len(hpi_filtered)}")
print(f"\nDate range: {hpi_filtered['Date'].min()} to {hpi_filtered['Date'].max()}")
print(f"\nRegions included:")
for r in sorted(hpi_filtered['Region_Name'].unique()):
    print(f"  {r}")
print(f"\nSample data:")
print(hpi_filtered.head(10).to_string(index=False))

Rows after filtering: 1170

Date range: 2015-01-01 00:00:00 to 2025-10-01 00:00:00

Regions included:
  East Midlands
  East of England
  London
  North East
  North West
  South East
  South West
  West Midlands Region
  Yorkshire and The Humber

Sample data:
      Date              Region_Name  Average_Price  year  month
2015-01-01               South East         266846  2015      1
2015-01-01               South West         205460  2015      1
2015-01-01 Yorkshire and The Humber         128167  2015      1
2015-01-01                   London         431270  2015      1
2015-01-01     West Midlands Region         151953  2015      1
2015-01-01          East of England         225847  2015      1
2015-01-01            East Midlands         147972  2015      1
2015-01-01               North East         110842  2015      1
2015-01-01               North West         126599  2015      1
2015-02-01               North East         112138  2015      2


In [4]:
# ── LOAD ASHE EARNINGS DATA ───────────────────────────────────────────────
import zipfile
import io

raw_path = '../data/raw/'

# Find all ASHE folders/zips
ashe_items = [f for f in os.listdir(raw_path) 
              if 'ashe' in f.lower() or 'table1' in f.lower()]
print(f"ASHE items found: {len(ashe_items)}")
for item in sorted(ashe_items):
    print(f"  {item}")

# Check what is inside one of them
sample = sorted(ashe_items)[0]
sample_path = os.path.join(raw_path, sample)
print(f"\nChecking contents of: {sample}")
print(f"Is directory: {os.path.isdir(sample_path)}")

if os.path.isdir(sample_path):
    contents = os.listdir(sample_path)
    print(f"Files inside:")
    for f in contents:
        print(f"  {f}")
elif sample.endswith('.zip'):
    with zipfile.ZipFile(sample_path) as z:
        print(f"Files inside zip:")
        for f in z.namelist():
            print(f"  {f}")

ASHE items found: 11
  ashetable12021revised
  ashetable12022revised
  ashetable12023revised
  ashetable12024revised
  ashetable12025provisional
  table12015revised
  table12016revised
  table12017revised
  table12018revised
  table12019revised
  table12020revised

Checking contents of: ashetable12021revised
Is directory: True
Files inside:
  Total Table 1.5a   Hourly pay - Gross 2021.xls
  Total Table 1.4b   Overtime pay 2021 CV.xls
  Total Table 1.6b   Hourly pay - Excluding overtime 2021 CV.xls
  Total Table 1.5b   Hourly pay - Gross 2021 CV.xls
  Total Table 1.2b   Weekly pay - Excluding overtime 2021 CV.xls
  Total Table 1.7a   Annual pay - Gross 2021.xls
  Total Table 1.10a   Paid hours worked - Basic 2021.xls
  Total Table 1.9b   Paid hours worked - Total 2021 CV.xls
  Total Table 1.3b   Basic Pay - Including other pay 2021 CV.xls
  Total Table 1.3a   Basic Pay - Including other pay 2021.xls
  Total Table 1.1b   Weekly pay - Gross 2021 CV.xls
  Total Table 1.7b   Annual pay - Gr

In [5]:
# ── LOAD ANNUAL PAY FROM EACH ASHE YEAR ──────────────────────────────────

ashe_dfs = []

for item in sorted(ashe_items):
    item_path = os.path.join(raw_path, item)
    
    # Find the annual pay gross file
    if os.path.isdir(item_path):
        files = os.listdir(item_path)
        target = [f for f in files if '1.7a' in f and 'Annual pay - Gross' in f]
        
        if target:
            file_path = os.path.join(item_path, target[0])
            print(f"Loading: {target[0]}")
            
            try:
                # Read the xls file
                df = pd.read_excel(file_path, sheet_name=None)
                sheet_names = list(df.keys())
                print(f"  Sheets: {sheet_names}")
            except Exception as e:
                print(f"  Error: {e}")
        else:
            print(f"No 1.7a file found in: {item}")

Loading: Total Table 1.7a   Annual pay - Gross 2021.xls
  Sheets: ['Notes', 'All', 'Male', 'Female', 'Full-Time', 'Part-Time', 'Male Full-Time', 'Male Part-Time', 'Female Full-Time', 'Female Part-Time']
Loading: Total Table 1.7a Annual pay - Gross 2022.xlsx
  Sheets: ['Notes', 'All', 'Male', 'Female', 'Full-Time', 'Part-Time', 'Male Full-Time', 'Male Part-Time', 'Female Full-Time', 'Female Part-Time']
Loading: Total Table 1.7a   Annual pay - Gross 2023.xlsx
  Sheets: ['Notes', 'All', 'Male', 'Female', 'Full-Time', 'Part-Time', 'Male Full-Time', 'Male Part-Time', 'Female Full-Time', 'Female Part-Time']
Loading: Total Table 1.7a   Annual pay - Gross 2024.xlsx
  Sheets: ['Notes', 'All', 'Male', 'Female', 'Full-Time', 'Part-Time', 'Male Full-Time', 'Male Part-Time', 'Female Full-Time', 'Female Part-Time']
Loading: PROV - Total Table 1.7a   Annual pay - Gross 2025.xlsx
  Sheets: ['Notes', 'All', 'Male', 'Female', 'Full-Time', 'Part-Time', 'Male Full-Time', 'Male Part-Time', 'Female Full-Tim

In [6]:
# ── INSPECT ONE SHEET TO UNDERSTAND STRUCTURE ─────────────────────────────

# Load one file to see what the Full-Time sheet looks like
sample_path = os.path.join(raw_path, 'ashetable12021revised',
              'Total Table 1.7a   Annual pay - Gross 2021.xls')

df_sample = pd.read_excel(sample_path, sheet_name='Full-Time', header=None)

print(f"Shape: {df_sample.shape}")
print(f"\nFirst 30 rows:")
print(df_sample.iloc[:30].to_string())

Shape: (14, 20)

First 30 rows:
                                                                                                                                                                                                           0     1           2       3           4      5           6            7        8        9        10       11       12       13       14       15       16  17                      18                                                          19
0                                                                                                                    Table 1.7a   Annual pay - Gross (£) - For full-time employee jobsa: United Kingdom, 2021   NaN         NaN     NaN         NaN    NaN         NaN          NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN NaN                     NaN                                                         NaN
1                                                                             

In [7]:
# ── CHECK ALL SHEET FOR REGIONAL DATA ────────────────────────────────────

df_all = pd.read_excel(sample_path, sheet_name='All', header=None)

print(f"Shape: {df_all.shape}")
print(f"\nAll rows:")
print(df_all.iloc[:, :5].to_string())

Shape: (14, 20)

All rows:
                                                                                                                                                                                                            0     1           2       3           4
0                                                                                                                          Table 1.7a   Annual pay - Gross (£) - For all employee jobsa: United Kingdom, 2021   NaN         NaN     NaN         NaN
1                                                                                                                                                                                                         NaN   NaN         NaN     NaN         NaN
2                                                                                                                                                                                                         NaN   NaN      Number     NaN      Annu

In [8]:
# ── CHECK ALL FILES IN THE 2021 FOLDER ───────────────────────────────────

folder_path = os.path.join(raw_path, 'ashetable12021revised')
all_files = os.listdir(folder_path)

print("All files in 2021 folder:")
for f in sorted(all_files):
    print(f"  {f}")

All files in 2021 folder:
  Total Table 1.10a   Paid hours worked - Basic 2021.xls
  Total Table 1.10b   Paid hours worked - Basic 2021 CV.xls
  Total Table 1.11a   Paid hours worked - Overtime 2021.xls
  Total Table 1.11b   Paid hours worked - Overtime 2021 CV.xls
  Total Table 1.12  Gender pay gap 2021.xls
  Total Table 1.1a   Weekly pay - Gross 2021.xls
  Total Table 1.1b   Weekly pay - Gross 2021 CV.xls
  Total Table 1.2a   Weekly pay - Excluding overtime 2021.xls
  Total Table 1.2b   Weekly pay - Excluding overtime 2021 CV.xls
  Total Table 1.3a   Basic Pay - Including other pay 2021.xls
  Total Table 1.3b   Basic Pay - Including other pay 2021 CV.xls
  Total Table 1.4a   Overtime pay 2021.xls
  Total Table 1.4b   Overtime pay 2021 CV.xls
  Total Table 1.5a   Hourly pay - Gross 2021.xls
  Total Table 1.5b   Hourly pay - Gross 2021 CV.xls
  Total Table 1.6a   Hourly pay - Excluding overtime 2021.xls
  Total Table 1.6b   Hourly pay - Excluding overtime 2021 CV.xls
  Total Table 1.7a

In [9]:
# ── INSPECT ONE TABLE 8 ZIP FILE ─────────────────────────────────────────

import zipfile

sample_zip = '../data/raw/ashe-table8-2021.zip'

with zipfile.ZipFile(sample_zip) as z:
    print("Files inside 2021 Table 8 zip:")
    for f in sorted(z.namelist()):
        print(f"  {f}")

IsADirectoryError: [Errno 21] Is a directory: '../data/raw/ashe-table8-2021.zip'

In [10]:
# ── CHECK WHAT IS INSIDE THE TABLE 8 FOLDER ──────────────────────────────

import os

sample_folder = '../data/raw/ashe-table8-2021.zip'

print(f"Is directory: {os.path.isdir(sample_folder)}")
print(f"\nContents:")
for f in sorted(os.listdir(sample_folder)):
    print(f"  {f}")

Is directory: True

Contents:
  Home Geography Table 8.10a   Paid hours worked - Basic 2021.xls
  Home Geography Table 8.10b   Paid hours worked - Basic 2021 CV.xls
  Home Geography Table 8.11a   Paid hours worked - Overtime 2021.xls
  Home Geography Table 8.11b   Paid hours worked - Overtime 2021 CV.xls
  Home Geography Table 8.12  Gender pay gap 2021.xls
  Home Geography Table 8.1a   Weekly pay - Gross 2021.xls
  Home Geography Table 8.1b   Weekly pay - Gross 2021 CV.xls
  Home Geography Table 8.2a   Weekly pay - Excluding overtime 2021.xls
  Home Geography Table 8.2b   Weekly pay - Excluding overtime 2021 CV.xls
  Home Geography Table 8.3a   Basic Pay - Including other pay 2021.xls
  Home Geography Table 8.3b   Basic Pay - Including other pay 2021 CV.xls
  Home Geography Table 8.4a   Overtime pay 2021.xls
  Home Geography Table 8.4b   Overtime pay 2021 CV.xls
  Home Geography Table 8.5a   Hourly pay - Gross 2021.xls
  Home Geography Table 8.5b   Hourly pay - Gross 2021 CV.xls
  Home

In [11]:
# ── INSPECT TABLE 8.7a STRUCTURE ─────────────────────────────────────────

file_path = '../data/raw/ashe-table8-2021.zip/Home Geography Table 8.7a   Annual pay - Gross 2021.xls'

df = pd.read_excel(file_path, sheet_name='Full-Time', header=None)

print(f"Shape: {df.shape}")
print(f"\nFirst 30 rows (first 5 columns):")
print(df.iloc[:30, :5].to_string())

Shape: (422, 20)

First 30 rows (first 5 columns):
                                                                                           0          1           2       3           4
0   Table 8.7a   Annual pay - Gross (£) - For full-time employee jobsa: United Kingdom, 2021        NaN         NaN     NaN         NaN
1                                                                                        NaN        NaN         NaN     NaN         NaN
2                                                                                        NaN        NaN      Number     NaN      Annual
3                                                                                        NaN        NaN    of jobsb     NaN  percentage
4                                                                                Description       Code  (thousand)  Median      change
5                                                                            United Kingdom   K02000001       17445   31224         N

In [12]:
# ── EXTRACT REGIONAL EARNINGS FROM ALL TABLE 8 YEARS ─────────────────────

# The 9 England region codes — all start with E12
england_region_codes = [
    'E12000001',  # North East
    'E12000002',  # North West
    'E12000003',  # Yorkshire and The Humber
    'E12000004',  # East Midlands
    'E12000005',  # West Midlands
    'E12000006',  # East of England
    'E12000007',  # London
    'E12000008',  # South East
    'E12000009',  # South West
]

ashe_folders = sorted([
    f for f in os.listdir('../data/raw/')
    if 'ashe-table8' in f
])

earnings_dfs = []

for folder in ashe_folders:
    # Extract year from folder name
    year = int(folder.replace('ashe-table8-', '').replace('.zip', ''))
    
    folder_path = f'../data/raw/{folder}'
    
    # Find the 8.7a annual pay file
    files = os.listdir(folder_path)
    target = [f for f in files if '8.7a' in f and 'Annual pay - Gross' in f]
    
    if not target:
        print(f"No 8.7a file found for {year}")
        continue
    
    file_path = os.path.join(folder_path, target[0])
    
    # Read the Full-Time sheet
    df = pd.read_excel(file_path, sheet_name='Full-Time', header=None)
    
    # Row 4 is the header row
    df.columns = df.iloc[4]
    df = df.iloc[5:].reset_index(drop=True)
    
    # Keep only Description, Code, Median columns
    df = df[['Description', 'Code', 'Median']].copy()
    df.columns = ['region_name', 'area_code', 'median_earnings']
    
    # Filter to England regions only
    df = df[df['area_code'].isin(england_region_codes)].copy()
    
    # Add year
    df['year'] = year
    
    # Convert median to numeric
    df['median_earnings'] = pd.to_numeric(df['median_earnings'], errors='coerce')
    
    earnings_dfs.append(df)
    print(f"Year {year}: {len(df)} regions loaded")

# Combine all years
earnings = pd.concat(earnings_dfs, ignore_index=True)

print(f"\nTotal rows: {len(earnings)}")
print(f"\nSample:")
print(earnings.head(15).to_string(index=False))

Year 2015: 0 regions loaded
Year 2016: 9 regions loaded
Year 2017: 9 regions loaded
Year 2018: 9 regions loaded
Year 2019: 9 regions loaded
Year 2020: 9 regions loaded
Year 2021: 9 regions loaded
Year 2022: 9 regions loaded
Year 2023: 9 regions loaded
Year 2024: 9 regions loaded
Year 2025: 9 regions loaded

Total rows: 90

Sample:
              region_name area_code  median_earnings  year
              North East  E12000001            25660  2016
              North West  E12000002            26220  2016
Yorkshire and The Humber  E12000003            25947  2016
           East Midlands  E12000004            26554  2016
           West Midlands  E12000005            26270  2016
                    East  E12000006            30000  2016
                  London  E12000007            33694  2016
              South East  E12000008            30741  2016
              South West  E12000009            26796  2016
              North East  E12000001            26046  2017
              Nort

In [13]:
# ── DEBUG 2015 FILE ───────────────────────────────────────────────────────

folder_path = '../data/raw/ashe-table8-2015.zip'
files = os.listdir(folder_path)

print("Files in 2015 folder:")
for f in sorted(files):
    print(f"  {f}")

Files in 2015 folder:
  Home Geography Table 8.10a   Paid hours worked - Basic 2015.xls
  Home Geography Table 8.10b   Paid hours worked - Basic 2015 CV.xls
  Home Geography Table 8.11a   Paid hours worked - Overtime 2015.xls
  Home Geography Table 8.11b   Paid hours worked - Overtime 2015 CV.xls
  Home Geography Table 8.1a   Weekly pay - Gross 2015.xls
  Home Geography Table 8.1b   Weekly pay - Gross 2015 CV.xls
  Home Geography Table 8.2a   Weekly pay - Excluding overtime 2015.xls
  Home Geography Table 8.2b   Weekly pay - Excluding overtime 2015 CV.xls
  Home Geography Table 8.3a   Basic Pay - Including other pay 2015.xls
  Home Geography Table 8.3b   Basic Pay - Including other pay 2015 CV.xls
  Home Geography Table 8.4a   Overtime pay 2015.xls
  Home Geography Table 8.4b   Overtime pay 2015 CV.xls
  Home Geography Table 8.5a   Hourly pay - Gross 2015.xls
  Home Geography Table 8.5b   Hourly pay - Gross 2015 CV.xls
  Home Geography Table 8.6a   Hourly pay - Excluding overtime 2015.

In [14]:
# ── INSPECT 2015 FILE STRUCTURE ───────────────────────────────────────────

file_path_2015 = '../data/raw/ashe-table8-2015.zip/Home Geography Table 8.7a   Annual pay - Gross 2015.xls'

df_2015 = pd.read_excel(file_path_2015, sheet_name='Full-Time', header=None)

print(f"Shape: {df_2015.shape}")
print(f"\nFirst 15 rows (first 5 columns):")
print(df_2015.iloc[:15, :5].to_string())

Shape: (442, 20)

First 15 rows (first 5 columns):
                                                                                           0          1           2       3           4
0   Table 8.7a   Annual pay - Gross (£) - For full-time employee jobsa: United Kingdom, 2015        NaN         NaN     NaN         NaN
1                                                                                        NaN        NaN         NaN     NaN         NaN
2                                                                                        NaN        NaN      Number     NaN      Annual
3                                                                                        NaN        NaN    of jobsb     NaN  percentage
4                                                                                Description       Code  (thousand)  Median      change
5                                                                             United Kingdom        NaN       16003   27615         1

In [15]:
# ── FIX 2015 — FILTER BY REGION NAME INSTEAD OF CODE ─────────────────────

england_region_names = [
    'North East',
    'North West',
    'Yorkshire and The Humber',
    'East Midlands',
    'West Midlands',
    'East',
    'London',
    'South East',
    'South West'
]

file_path_2015 = '../data/raw/ashe-table8-2015.zip/Home Geography Table 8.7a   Annual pay - Gross 2015.xls'

df_2015 = pd.read_excel(file_path_2015, sheet_name='Full-Time', header=None)

# Set headers
df_2015.columns = df_2015.iloc[4]
df_2015 = df_2015.iloc[5:].reset_index(drop=True)
df_2015 = df_2015[['Description', 'Code', 'Median']].copy()
df_2015.columns = ['region_name', 'area_code', 'median_earnings']

# Strip whitespace
df_2015['region_name'] = df_2015['region_name'].astype(str).str.strip()

# Filter by name
df_2015_regions = df_2015[df_2015['region_name'].isin(england_region_names)].copy()
df_2015_regions['year'] = 2015
df_2015_regions['median_earnings'] = pd.to_numeric(
    df_2015_regions['median_earnings'], errors='coerce')

print(f"2015 regions found: {len(df_2015_regions)}")
print(df_2015_regions.to_string(index=False))

2015 regions found: 9
             region_name area_code  median_earnings  year
              North East       NaN            25232  2015
              North West       NaN            25711  2015
Yorkshire and The Humber       NaN            25114  2015
           East Midlands       NaN            25609  2015
           West Midlands       NaN            25598  2015
                    East       NaN            29259  2015
                  London       NaN            33109  2015
              South East       NaN            30074  2015
              South West       NaN            26496  2015


In [16]:
# ── ADD AREA CODES TO 2015 AND COMBINE ALL YEARS ─────────────────────────

# Map region names to area codes for 2015
region_code_map = {
    'North East':                 'E12000001',
    'North West':                 'E12000002',
    'Yorkshire and The Humber':   'E12000003',
    'East Midlands':              'E12000004',
    'West Midlands':              'E12000005',
    'East':                       'E12000006',
    'London':                     'E12000007',
    'South East':                 'E12000008',
    'South West':                 'E12000009',
}

df_2015_regions['area_code'] = df_2015_regions['region_name'].map(region_code_map)

# Combine 2015 with the rest
earnings_all = pd.concat([earnings, df_2015_regions], ignore_index=True)

# Sort by region and year
earnings_all = earnings_all.sort_values(['area_code', 'year']).reset_index(drop=True)

# Standardise region names — rename East to East of England
earnings_all['region_name'] = earnings_all['region_name'].str.strip()
earnings_all['region_name'] = earnings_all['region_name'].replace(
    {'East': 'East of England',
     'West Midlands': 'West Midlands Region'})

print(f"Total rows: {len(earnings_all)}")
print(f"Years covered: {sorted(earnings_all['year'].unique())}")
print(f"\nRegions:")
for r in sorted(earnings_all['region_name'].unique()):
    print(f"  {r}")

print(f"\nSample — London earnings over time:")
london = earnings_all[earnings_all['area_code'] == 'E12000007'][
    ['year', 'region_name', 'median_earnings']]
print(london.to_string(index=False))

Total rows: 99
Years covered: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Regions:
  East Midlands
  East of England
  London
  North East
  North West
  South East
  South West
  West Midlands Region
  Yorkshire and The Humber

Sample — London earnings over time:
 year region_name  median_earnings
 2015      London            33109
 2016      London            33694
 2017      London            34725
 2018      London            35702
 2019      London            36851
 2020      London            38526
 2021      London            37635
 2022      London            39497
 2023      London            42083
 2024      London            44796
 2025      London            46414


### Data Loading Summary

**Dataset 1 — UK House Price Index (HM Land Registry)**
- Source: HM Land Registry via GOV.UK
- Coverage: January 2015 to October 2025
- Filtered to: 9 England regions only
- Rows: 1,170 (9 regions × ~130 months)
- Licence: Open Government Licence v3.0

**Dataset 2 — ASHE Table 8 Regional Earnings (ONS)**
- Source: Office for National Statistics
- Coverage: 2015 to 2025 (annual, April each year)
- Filtered to: 9 England regions, full-time employees, median gross annual earnings
- Rows: 99 (9 regions × 11 years)
- Licence: Open Government Licence v3.0

**Note on methodology:**
The ASHE earnings data is collected every April. To create an annual
affordability ratio we use the December house price for each year
matched to the April earnings for the same year. This is a standard
approach used by ONS and the Office for Budget Responsibility in
affordability analysis.

In [17]:
# ── MERGE HOUSE PRICE AND EARNINGS DATA ───────────────────────────────────

# For each year, use December house price (end of year snapshot)
# matched to April earnings (ASHE reference date)

hpi_december = hpi_filtered[hpi_filtered['month'] == 12].copy()

# Standardise region names to match earnings data
name_map = {
    'West Midlands Region': 'West Midlands Region',
    'Yorkshire and The Humber': 'Yorkshire and The Humber',
    'East of England': 'East of England',
    'North East': 'North East',
    'North West': 'North West',
    'East Midlands': 'East Midlands',
    'London': 'London',
    'South East': 'South East',
    'South West': 'South West'
}

hpi_december['Region_Name'] = hpi_december['Region_Name'].map(name_map)

# Merge on region name and year
merged = pd.merge(
    hpi_december[['year', 'Region_Name', 'Average_Price']],
    earnings_all[['year', 'region_name', 'median_earnings', 'area_code']],
    left_on=['year', 'Region_Name'],
    right_on=['year', 'region_name'],
    how='inner'
)

# Drop duplicate region name column
merged = merged.drop(columns=['region_name'])
merged = merged.rename(columns={'Region_Name': 'region'})

# Create the affordability ratio
merged['affordability_ratio'] = (
    merged['Average_Price'] / merged['median_earnings']
).round(2)

print(f"Merged rows: {len(merged)}")
print(f"Years: {sorted(merged['year'].unique())}")
print(f"\nSample:")
print(merged.sort_values(['region', 'year']).head(15).to_string(index=False))

Merged rows: 90
Years: [np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]

Sample:
 year          region  Average_Price  median_earnings area_code  affordability_ratio
 2015   East Midlands         158962            25609 E12000004                 6.21
 2016   East Midlands         168191            26554 E12000004                 6.33
 2017   East Midlands         178086            26556 E12000004                 6.71
 2018   East Midlands         184588            27606 E12000004                 6.69
 2019   East Midlands         187279            28556 E12000004                 6.56
 2020   East Midlands         201798            29417 E12000004                 6.86
 2021   East Midlands         218318            29161 E12000004                 7.49
 2022   East Midlands         237956            30946 E12000004                 7.69
 2023   East Midlands         229757      

In [18]:
# ── SAVE MERGED DATASET ───────────────────────────────────────────────────

import os
os.makedirs('../data/processed/', exist_ok=True)

merged.to_csv('../data/processed/uk_housing_affordability.csv', index=False)

print("=" * 55)
print("NOTEBOOK 01 — DATA LOADING SUMMARY")
print("=" * 55)
print(f"House price data:       1,170 rows (9 regions, 2015-2025)")
print(f"Earnings data:          99 rows (9 regions, 2015-2025)")
print(f"Merged dataset:         {len(merged)} rows")
print(f"Years covered:          2015 to 2024")
print(f"Regions covered:        {merged['region'].nunique()}")
print(f"Affordability ratio:    min {merged['affordability_ratio'].min()}")
print(f"                        max {merged['affordability_ratio'].max()}")
print(f"Saved to:               data/processed/uk_housing_affordability.csv")
print("=" * 55)
print("NEXT: Notebook 02 — Exploratory Data Analysis")
print("=" * 55)

NOTEBOOK 01 — DATA LOADING SUMMARY
House price data:       1,170 rows (9 regions, 2015-2025)
Earnings data:          99 rows (9 regions, 2015-2025)
Merged dataset:         90 rows
Years covered:          2015 to 2024
Regions covered:        9
Affordability ratio:    min 4.49
                        max 15.01
Saved to:               data/processed/uk_housing_affordability.csv
NEXT: Notebook 02 — Exploratory Data Analysis
